# TP0+ — <!-- BEGIN SOLUTION -->SOLUTION (instructors only) — <!-- END SOLUTION -->xarray basics: open a netCDF file, select, average, and draw maps

**Suggested duration: ~1h30–2h** · **Prerequisites**: TP0 (you can log in to Kabre and open a notebook with the `psf2026` kernel).

Before TP1, this notebook walks you **step by step** through the handful of Python commands you will use all week to work with ocean data:
open a netCDF file, look inside it, pick the part you need, average it, and draw clean maps with land and coastlines.
Most of the code is already written: **run it, read it, change it**. The few `# TODO` cells are short.

## Learning objectives
By the end of this notebook you should be able to:
1. Open a netCDF file with **xarray** and find its dimensions, coordinates, variables and units.
2. Select data **by index** (`isel`) and **by value** (`sel`): a date, a point, a sub-region, a period.
3. Reduce data: a time mean, a monthly climatology.
4. Draw a **map** with **cartopy**: projection, land, coastlines, grid labels, colorbar.
5. Put several maps side by side with **`subplots`**, with a **shared color scale**, and save the figure.

## How to work through this notebook
- Cells marked `# TODO` contain code **you need to complete** (replace the `...`). Everything else is already written: run it and read it.
- After each exercise, a **✅ Check** box tells you what a correct result should look like.
- **🚀 Advanced track** blocks are optional: do them only if you finished the main exercise.
- Stuck on a `# TODO`? Below each exercise, a **💡 Hint** box (and, as a last resort, a **🔑 Solution** box) can be opened with a click. Try for a few minutes before opening them!
- Stuck for more than 10 minutes? Ask a trainer — that's what we're here for.

> **⚠️ Before you start: work on your own copy.** In the JupyterLab file browser, right-click this notebook → *Duplicate*, then rename the copy to `00_basics_xarray_EN_mine.ipynb` and open it. Work only in your `_mine` copy, so the original stays clean and you can always go back to it.

---
## 0. Setup — imports and paths

Run this cell as is. It imports the libraries and defines:
- `OBS_DIR`: where the observation files are stored on Kabre (read-only, shared by everyone);
- `SST_FILE`: the file used today, **daily satellite SST** (OSTIA, 0.05°) for 2023 around Costa Rica (4–14°N, 92–77°W: Pacific **and** Caribbean sides);
- `WORK_DIR`: a folder `data_tp0/`, created next to this notebook, where **your** figures go.

| Library | What it does |
|---|---|
| `xarray` (`xr`) | reads netCDF files and handles labelled arrays (dimensions with names and coordinates) |
| `numpy` (`np`) | basic numerical arrays and maths |
| `matplotlib.pyplot` (`plt`) | figures and plots |
| `cartopy` (`ccrs`, `cfeature`) | map projections, coastlines, land, borders |

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Observation files already on Kabre (read-only)
OBS_DIR = "/data/gcambon/COMMONDATA/DATASETS_PSF/"
if not Path(OBS_DIR).exists():          # not on Kabre: instructor's local copy
    OBS_DIR = "../data/DATASETS_PSF"

SST_FILE = f"{OBS_DIR}/COSTARICA_DS/sst_costarica_2023.nc"

# Your own output files go here
WORK_DIR = Path("data_tp0")
WORK_DIR.mkdir(exist_ok=True)

print("SST file found:", Path(SST_FILE).exists())

**✅ Check**: the cell prints `SST file found: True`. If it prints `False`, check that you are running the notebook on Kabre with the `psf2026` kernel (top right of the notebook), and call a trainer.

---
## 1. Open a netCDF file and look inside

`xr.open_dataset()` opens the file and returns a **Dataset**: a container holding several variables that share the same dimensions and coordinates.
It is **lazy**: at this point only the description of the file (the *header*) is read, not the data itself, so it is instant even for a big file.

Displaying the Dataset (just write its name as the last line of a cell) shows, from top to bottom:
- **Dimensions**: the names and sizes of the axes (here `time`, `latitude`, `longitude`);
- **Coordinates**: the values along each axis (the dates, the latitudes, the longitudes);
- **Data variables**: the physical fields (here `analysed_sst`), with the dimensions they depend on;
- **Attributes**: metadata about the whole file (who made it, how, from which satellites…).

Click on the small icons on the right of each line to unfold the attributes or preview the values.

In [ ]:
ds = xr.open_dataset(SST_FILE)
ds

The same information, piece by piece:

In [ ]:
print("Dimensions :", dict(ds.sizes))
print("Coordinates:", list(ds.coords))
print("Variables  :", list(ds.data_vars))
print("First / last date:", ds["time"].values[0], "/", ds["time"].values[-1])
print("Latitudes  :", float(ds["latitude"][0]), "->", float(ds["latitude"][-1]))
print("Longitudes :", float(ds["longitude"][0]), "->", float(ds["longitude"][-1]))

### Exercise 1 — a variable and its attributes

`ds["analysed_sst"]` extracts one variable: a **DataArray** (a single array, with its dimensions, coordinates and attributes).
Its attributes are in the dictionary `.attrs`: for example `.attrs["long_name"]`.

Complete the cell to print the **units** and the **long name** of the SST, and the **number of time steps**.

In [ ]:
sst_k = ds["analysed_sst"]
### STUDENT: # TODO: print the units, the long name, and the number of time steps of sst_k
### STUDENT: print("Units    :", ...)
### STUDENT: print("Long name:", ...)
### STUDENT: print("Time steps:", ...)
### BEGIN SOLUTION
print("Units    :", sst_k.attrs["units"])
print("Long name:", sst_k.attrs["long_name"])
print("Time steps:", sst_k.sizes["time"])
### END SOLUTION
### HINT: The units are in `sst_k.attrs["units"]`; the long name follows the same pattern.
### HINT: The number of time steps is the size of the `time` dimension: `sst_k.sizes["time"]` (or `len(sst_k["time"])`).

**✅ Check**: units `kelvin`, long name `analysed sea surface temperature`, **365** time steps (one map per day of 2023).

> 💬 Always look at the units before plotting. Satellite SST is very often stored in **kelvin**: 300 K ≈ 27 °C.

---
## 2. From kelvin to °C

Arithmetic on a DataArray applies to every value and keeps the dimensions and coordinates.
**But** the attributes are dropped (xarray cannot know whether `units` is still right after your operation), so we set them again ourselves.

In [ ]:
sst = sst_k - 273.15
sst.attrs = {"long_name": "SST", "units": "°C"}
sst

> 💬 This time you see **numbers**, not just a description: `open_dataset` only read the header (section 1), but an arithmetic operation like `- 273.15` needs the values, so xarray reads the **whole variable** into memory (≈ 175 MB here, no problem).
> On a big file (a multi-year 3D model output can weigh tens of GB), **select first** (a region, a period, a depth) **and compute after**: `ds["var"].sel(...)` stays cheap, and only the selected part is read.

---
## 3. Select the part you need: `isel` and `sel`

Two ways to select, with the **name** of the dimension (never "axis 0, axis 1" as in numpy):

| Method | Selects by… | Example |
|---|---|---|
| `isel` | **position** (integer index, starts at 0) | `sst.isel(time=0)` → the first day |
| `sel` | **value** of the coordinate | `sst.sel(time="2023-03-15")` → 15 March 2023 |
| `sel(..., method="nearest")` | closest value | `sst.sel(latitude=10.0, longitude=-87.0, method="nearest")` |
| `sel` + `slice(a, b)` | a range of values (**both ends included**) | `sst.sel(latitude=slice(8, 12))` |

A selection on one value **removes** that dimension: one day → a 2D map (lat, lon); one point → a 1D time series.

In [ ]:
day0 = sst.isel(time=0)                     # first day, by index
day = sst.sel(time="2023-03-15")            # one day, by date
print(day0.dims, "->", str(day0["time"].values)[:10])
print(day.dims, "->", str(day["time"].values)[:10])

day.plot(cmap="turbo")                      # xarray quick look: one line, no map yet
plt.title("SST on 15 March 2023 (quick look)")
plt.show()

The white areas are **land** (no SST there: the values are `NaN`, *Not a Number*). The Pacific is on the left, the **Caribbean Sea** on the top right. The small coloured patch inland is **Lake Nicaragua**: OSTIA also analyses large lakes. We will draw proper coastlines (and hide the lake under the land) in section 5.

### Exercise 2 — a time series at one point

Extract the daily SST time series at **11°N, 87°W** (offshore of the Gulf of Papagayo, north-west Costa Rica) and plot it.
Remember: west longitudes are **negative**, and the exact point is probably not on the grid → use `method="nearest"`.

In [ ]:
### STUDENT: # TODO: select the point 11°N, 87°W (nearest grid point)
### STUDENT: ts = ...
### BEGIN SOLUTION
ts = sst.sel(latitude=11.0, longitude=-87.0, method="nearest")
### END SOLUTION

print(f"Grid point used: {float(ts.latitude):.3f}°N, {float(ts.longitude):.3f}°E (negative = west)")
fig, ax = plt.subplots(figsize=(10, 3.5))
ts.plot(ax=ax)
ax.set_title("Daily SST at 11°N, 87°W — 2023")
ax.grid(alpha=0.3)
plt.show()

print(f"Min: {float(ts.min()):.2f} °C on {str(ts.idxmin().values)[:10]}")
print(f"Max: {float(ts.max()):.2f} °C on {str(ts.idxmax().values)[:10]}")
### HINT: `sst.sel(latitude=..., longitude=..., method="nearest")` — longitude is **-87.0**, not 87.

**✅ Check**: grid point ≈ 11.025°N, -86.975°E. The SST drops sharply in **January–March**, down to **≈ 23 °C around 20 February**, then stays near **29–30 °C** from May to October.
That winter cooling is the **Papagayo upwelling**: strong trade winds blow through a gap in the mountains of Central America and push the warm surface water offshore, bringing cold water up.
If you get `nan` everywhere, you probably typed `longitude=87.0` (a point on land… on the other side of the planet!) or picked a point on land.

### Exercise 3 — a sub-region and a period

With `slice`, select the box **8–12°N, 88–84°W** for the month of **February 2023** only.
A string like `"2023-02"` selects a whole month; `slice("2023-02-01", "2023-02-28")` would do the same.

In [ ]:
### STUDENT: # TODO: select 8–12°N, 88–84°W, February 2023
### STUDENT: box_feb = sst.sel(latitude=..., longitude=..., time=...)
### BEGIN SOLUTION
box_feb = sst.sel(latitude=slice(8, 12), longitude=slice(-88, -84), time="2023-02")
### END SOLUTION
print(dict(box_feb.sizes))
### HINT: `latitude=slice(8, 12)`, and for the longitude the smallest (most negative) value comes first: `slice(-88, -84)`.

**✅ Check**: `time: 28`, `latitude: 80`, `longitude: 80` (4° / 0.05° = 80 points in each direction).

### ⚠️ The latitude-order trap

`slice(a, b)` follows the **order in which the coordinate is stored**. In this file the latitude is stored **increasing** (from 4°N to 14°N), so `slice(8, 12)` works.
Some products (many reanalyses, some satellite files) store the latitude **decreasing** (from north to south). Then `slice(8, 12)` returns an **empty selection, without any error**, and all your means become `nan`.
See for yourself:

In [ ]:
wrong = sst.sel(latitude=slice(12, 8))      # reversed slice on an increasing coordinate
print("Size of the wrong selection:", wrong.sizes["latitude"], "latitudes")

# The habit to take: check the order BEFORE slicing
lat = ds["latitude"]
print("Latitude increasing?", bool(lat[0] < lat[-1]))

**✅ Check**: the wrong selection has **0** latitudes, and no error was raised. Rule of thumb: always check `ds.latitude[0]` and `ds.latitude[-1]` on a new file. If the latitude is decreasing, write `slice(12, 8)`, or sort it once with `ds = ds.sortby("latitude")`.

---
## 4. Reduce: means over a dimension

`.mean("time")` averages along `time`: 365 daily maps → **one** mean map. Same idea for `.min()`, `.max()`, `.std()`, `.sum()`… and for any dimension (`.mean(["latitude", "longitude"])` gives a time series averaged over the box).
Missing values (`NaN`, land) are **skipped** automatically.

`.groupby("time.month").mean()` puts the days into 12 groups (one per month) and averages each group: a **monthly climatology** with a new dimension `month` (1 to 12).

> ⏳ The data is already in memory since section 2, but averaging 365 maps still takes a few seconds.

In [ ]:
sst_mean = sst.mean("time")                       # annual mean map  (latitude, longitude)
sst_monthly = sst.groupby("time.month").mean()    # 12 monthly maps  (month, latitude, longitude)
print(sst_mean.dims, sst_monthly.dims)
print(f"Annual mean SST: from {float(sst_mean.min()):.1f} to {float(sst_mean.max()):.1f} °C")

### Exercise 4 — the mean of the box, computed two ways

Compute the **mean SST over the whole year** in the February box of exercise 3 (8–12°N, 88–84°W), first with **xarray**, then with **numpy** on the raw values (`.values` gives a plain numpy array; `np.nanmean` ignores the `NaN`s).
Computing the same number by two independent methods is a habit of scientific rigour: if they disagree, one of them is wrong.

In [ ]:
box = sst.sel(latitude=slice(8, 12), longitude=slice(-88, -84))     # whole year this time

### STUDENT: # TODO: the mean of `box` with xarray, then with numpy
### STUDENT: val_xr = ...
### STUDENT: val_np = ...
### BEGIN SOLUTION
val_xr = float(box.mean())
val_np = np.nanmean(box.values)
### END SOLUTION
print(f"xarray: {val_xr:.4f} °C   numpy: {val_np:.4f} °C   difference: {abs(val_xr - val_np):.1e}")
### HINT: `box.mean()` without any dimension averages over **all** dimensions; wrap it in `float(...)` to get a plain number.
### HINT: numpy: `np.nanmean(box.values)`.

**✅ Check**: both values ≈ **29.11 °C**, difference ~1e-14 or 0 (rounding noise only).

<details><summary>💬 More details (optional): should all grid points count the same?</summary>

This is a simple mean of grid points. On a large region, points near the poles cover a smaller area and should weigh less: `box.weighted(np.cos(np.deg2rad(box.latitude))).mean()`. Over a few degrees near the Equator, as here, the difference is negligible.

</details>

---
## 5. A real map with cartopy

The quick look `.plot()` of section 3 has no coastlines and no geographic axes. With **cartopy** we get a real map. The recipe:

1. `plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})` creates a **map axis** instead of an ordinary x/y axis.
2. `.plot(ax=ax, transform=ccrs.PlateCarree())` draws the field on that map. `transform` says that the data is given in longitude/latitude: always keep it.
3. `ax.add_feature(cfeature.LAND)` fills the land, `ax.coastlines()` draws the coastline, `cfeature.BORDERS` the country borders.
4. `ax.gridlines(draw_labels=True)` adds the latitude/longitude labels.

<details><summary>💬 More details (optional)</summary>

- `subplot_kw` (*subplot keywords*) is a dictionary of options passed to **each** panel the figure creates; here it says "make this panel a map, in this projection". Without it you get an ordinary axis that knows nothing about coastlines or projections.
- `PlateCarree` is the simplest projection: longitude on x, latitude on y. With any other map projection (e.g. `ccrs.Mercator()`), forgetting `transform=ccrs.PlateCarree()` puts the data in the wrong place.
- `"50m"` is the resolution of the Natural Earth coastlines (`"110m"` coarse, `"50m"` medium, `"10m"` fine): `"50m"` suits a region the size of Central America.
- `zorder` sets what is drawn on top: higher numbers are drawn above (land above the SST field, coastline above the land).

</details>

In [ ]:
proj = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=(9, 6), subplot_kw={"projection": proj})
sst_mean.plot(ax=ax, transform=proj, cmap="turbo",
              cbar_kwargs={"label": "SST (°C)", "shrink": 0.8})
ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.85", zorder=2)
ax.coastlines(resolution="50m" , linewidth=0.8, zorder=3)
ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.5, linestyle=":", zorder=3)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
gl.top_labels = gl.right_labels = False
ax.set_title("Mean SST 2023 — OSTIA")
plt.show()

**✅ Check**: grey land with Costa Rica, Nicaragua and Panama outlined; the SST field stops exactly at the coastline. Annual mean values between **≈ 27 °C and 30 °C**: warmest (> 29.5 °C) off El Salvador/Nicaragua in the north-west and along the Pacific coast of southern Costa Rica and western Panama; coolest right at the Gulf of Papagayo (≈ 86°W, 11°N, down to 27 °C) and in the Gulf of Panama (≈ 79°W, 7°N, ≈ 28 °C), the two upwelling areas. The Caribbean side is fairly uniform, ≈ 28.5–29 °C.

<!-- BEGIN SOLUTION -->> 🛠️ *Instructors*: the first `coastlines`/`add_feature` call downloads Natural Earth data from the internet. If the Kabre compute nodes have no internet access, the data must be pre-downloaded in the shared cartopy data directory (`cartopy.config["data_dir"]`).<!-- END SOLUTION -->

### Exercise 5 — zoom on Costa Rica, and choose the colour range

The recipe of section 5 is copied below, with **three blanks `___`** to fill, to map the **February** monthly mean on a zoom:
- `sst_monthly.sel(month=___)`: which month? (1 = January, 2 = February…)
- `vmin=___, vmax=___`: the colour range, here **24 to 30 °C**;
- `ax.set_extent([west, east, south, north], crs=proj)`: the zoom, here **88–82°W, 8–12°N** (west longitudes are negative!).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={"projection": proj})
### STUDENT: # TODO: fill in the blanks ___
### STUDENT: sst_monthly.sel(month=___).plot(ax=ax, transform=proj, cmap="turbo", vmin=___, vmax=___)
### BEGIN SOLUTION
sst_monthly.sel(month=2).plot(ax=ax, transform=proj, cmap="turbo", vmin=24, vmax=30)
### END SOLUTION
ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.85", zorder=2)
ax.coastlines(resolution="50m", linewidth=0.8, zorder=3)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
gl.top_labels = gl.right_labels = False
### STUDENT: ax.set_extent([___, ___, ___, ___], crs=proj)    # west, east, south, north
### BEGIN SOLUTION
ax.set_extent([-88, -82, 8, 12], crs=proj)    # west, east, south, north
### END SOLUTION
ax.set_title("Mean SST — February 2023")
plt.show()
### HINT: February is month `2`; the colour range is `vmin=24, vmax=30`.
### HINT: 88°W is `-88` and 82°W is `-82`: `ax.set_extent([-88, -82, 8, 12], crs=proj)`.

**✅ Check**: a tongue of cold water (**24 °C and below**, darkest colour: the arrows at the ends of the colorbar mean some values are outside 24–30) spreads westward from the Gulf of Papagayo along ≈ 10.5°N, while the Pacific coast further south stays above 29 °C. The Caribbean side (top right) is ≈ 27 °C.

---
## 6. Several maps side by side: `subplots`

To compare months, put several maps in one figure:
- `plt.subplots(2, 2, subplot_kw={"projection": proj})` returns a figure and a **2×2 array of map axes**; `axes.flat` lets you loop over them one by one.
- **Same `vmin`/`vmax` on every panel**, otherwise each map has its own colour scale and the colours cannot be compared from one panel to the next.
- `add_colorbar=False` on each panel, then **one shared colorbar** for the whole figure: `fig.colorbar(im, ax=axes, ...)`, where `im` is what the last `.plot()` returned.

<details><summary>💬 How does this loop work? (optional)</summary>

`zip(...)` walks through several lists **in parallel** and gives one item of each at every turn:

| turn | `ax` (from `axes.flat`) | `month` (from `[2, 8]`) | `name` (from `["February", "August"]`) |
|---|---|---|---|
| 1 | left panel | `2` | `"February"` |
| 2 | right panel | `8` | `"August"` |

`axes.flat` goes through the panels one by one (needed for a 2×2 grid, where `axes` is a 2D array: looping directly on it would give you **rows** of panels). `zip` stops at the shortest list: with 4 panels but only 3 months, the 4th panel stays empty, without any error.

Without `zip`, the same loop reads:
```python
axes_list = list(axes.flat)
for i in range(2):
    ax, month, name = axes_list[i], [2, 8][i], ["February", "August"][i]
```

</details>

Here is a 1×2 example (February vs August) to start from:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), subplot_kw={"projection": proj})

for ax, month, name in zip(axes.flat, [2, 8], ["February", "August"]):
    im = sst_monthly.sel(month=month).plot(ax=ax, transform=proj, cmap="turbo",
                                           vmin=22, vmax=32, add_colorbar=False)
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.85", zorder=2)
    ax.coastlines(resolution="50m", linewidth=0.8, zorder=3)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
    gl.top_labels = gl.right_labels = False
    ax.set_title(name)

fig.colorbar(im, ax=axes, orientation="horizontal", shrink=0.6, pad=0.12, label="SST (°C)")
fig.suptitle("Monthly mean SST 2023")
plt.show()

### Exercise 6 — one map per season

The cell below makes a **2×2** figure with the monthly means of **February, May, August and November** (one per season), with a shared colour scale (22–32 °C) and a single colorbar, then saves it as a PNG file (you will need this for the slides on Friday!).
Almost everything is written. Fill the **two blanks `___`**:
- which month to select at each turn of the loop (look at the variables of the `for` line);
- the name of the image file: `sst_4months.png`.

In [ ]:
months = [2, 5, 8, 11]
names = ["February", "May", "August", "November"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), subplot_kw={"projection": proj})

for ax, month, name in zip(axes.flat, months, names):
### STUDENT:     # TODO: which month should be selected at this turn of the loop?
### STUDENT:     im = sst_monthly.sel(month=___).plot(ax=ax, transform=proj, cmap="turbo",
### BEGIN SOLUTION
    im = sst_monthly.sel(month=month).plot(ax=ax, transform=proj, cmap="turbo",
### END SOLUTION
                                           vmin=22, vmax=32, add_colorbar=False)
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.85", zorder=2)
    ax.coastlines(resolution="50m", linewidth=0.8, zorder=3)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
    gl.top_labels = gl.right_labels = False
    ax.set_title(name)

fig.colorbar(im, ax=axes, orientation="horizontal", shrink=0.6, pad=0.06, label="SST (°C)")
fig.suptitle("Monthly mean SST 2023")
### STUDENT: # TODO: give the file name
### STUDENT: fig.savefig(WORK_DIR / "___", dpi=150, bbox_inches="tight")    # always BEFORE plt.show()
### BEGIN SOLUTION
fig.savefig(WORK_DIR / "sst_4months.png", dpi=150, bbox_inches="tight")    # always BEFORE plt.show()
### END SOLUTION
plt.show()
print("Saved:", (WORK_DIR / "sst_4months.png").exists())
### HINT: At each turn of the loop, the variable `month` takes the next value of `months` (2, then 5, 8, 11): use it in `.sel(month=month)`.
### HINT: The file name is a string: `"sst_4months.png"`.

**✅ Check**: four maps with the **same colours for the same temperature**, and one colorbar from 22 to 32 °C. February is clearly the coldest on the Pacific side (Papagayo and Panama upwellings in blue), and the Caribbean is also cooler (≈ 27 °C); May is the warmest on the Pacific side; August and November are warm and fairly uniform. The cell prints `Saved: True`, and `sst_4months.png` appears in the `data_tp0/` folder (file browser on the left).
If your image file is blank, you called `fig.savefig()` **after** `plt.show()`.

---
## 7. 🐞 Debugging exercise

A colleague wants to compare January and March side by side, and sends you this cell: *"it crashes, and I'm not sure the colours are right either"*.
It contains **two bugs**: one raises an error, the other is **silent** (the figure looks fine, but it is misleading).

Run it, **read the last line of the error message**, fix the first bug, run again, then look carefully at the figure to find the second one.

In [ ]:
### STUDENT: # 🐞 Fix me! (2 bugs)
### STUDENT: fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
### BEGIN SOLUTION
# Fixed version
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), subplot_kw={"projection": proj})
### END SOLUTION
for ax, month in zip(axes, [1, 3]):
### STUDENT:     sst_monthly.sel(month=month).plot(ax=ax, transform=proj, cmap="turbo")
### BEGIN SOLUTION
    sst_monthly.sel(month=month).plot(ax=ax, transform=proj, cmap="turbo", vmin=22, vmax=32)
### END SOLUTION
    ax.coastlines(resolution="50m")
    ax.set_title(f"Month {month}")
plt.show()
### BEGIN SOLUTION

# Bug 1: ValueError: Axes should be an instance of GeoAxes, got <class '...Axes'>
#        (raised by transform=proj; without it, the next error would be
#        AttributeError: 'Axes' object has no attribute 'coastlines')
#        -> plt.subplots() without subplot_kw={"projection": ...} creates ordinary axes,
#           not map axes: only a GeoAxes knows about coastlines, land, gridlines…
# Bug 2 (silent): no vmin/vmax, so each panel has its own colour range
#        (January ≈ 24.7-29.3 °C, March ≈ 24.2-30.2 °C): the same colour means different
#        temperatures on the two maps. Fix: the same vmin/vmax on both panels
#        (and ideally a single shared colorbar, as in section 6).
### END SOLUTION
### NO REVEAL
### HINT: Bug 1: which kind of axis knows how to draw coastlines? Compare with how `fig, axes` is created in section 6.
### HINT: Bug 2: read the numbers on the two colorbars. Does dark red mean the same temperature on both maps?

**✅ Check**: the fixed cell draws two maps with coastlines, and **both colorbars show exactly the same range**.

---
### 🚀 Advanced track A — a reusable map function and the 12-month climatology

You keep copying the same 6 lines (land, coastline, gridlines…). Write a function `plot_map(da, ax, title, **kwargs)` that draws `da` on `ax` with land, coastline and gridlines, passes `**kwargs` (e.g. `vmin`, `vmax`, `cmap`) to `.plot()`, and returns what `.plot()` returns.
Then use it to draw the **12 monthly means** on a **3×4** grid with one shared colorbar.

In [ ]:
### STUDENT: # 🚀 Your code here
### BEGIN SOLUTION
def plot_map(da, ax, title="", **kwargs):
    """Map of a 2D (latitude, longitude) DataArray on a cartopy GeoAxes."""
    kwargs.setdefault("cmap", "turbo")
    im = da.plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, **kwargs)
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.85", zorder=2)
    ax.coastlines(resolution="50m", linewidth=0.6, zorder=3)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {"size": 7}
    ax.set_title(title)
    return im


month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
fig, axes = plt.subplots(3, 4, figsize=(16, 11), subplot_kw={"projection": proj})
for ax, month in zip(axes.flat, range(1, 13)):
    im = plot_map(sst_monthly.sel(month=month), ax, month_names[month - 1], vmin=22, vmax=32)
fig.colorbar(im, ax=axes, orientation="horizontal", shrink=0.5, pad=0.05, label="SST (°C)")
fig.suptitle("Monthly mean SST 2023 — OSTIA")
plt.show()
### END SOLUTION
### HINT: Start with `def plot_map(da, ax, title="", **kwargs):` and move the land / coastline / gridlines lines inside it; return what `da.plot(...)` returns.
### HINT: Then `plt.subplots(3, 4, ...)` and loop with `zip(axes.flat, range(1, 13))`.

**✅ Check**: 12 panels, one colorbar. The cold upwelling tongues start in **December–January**, are strongest in **February–March**, and are gone by **May**.

### 🚀 Advanced track B — anomaly maps with a diverging colormap

An **anomaly** is the difference to a reference: here, `sst_monthly - sst_mean` (each monthly map minus the annual mean map; xarray aligns the coordinates and broadcasts over `month` for you).
Map the **March** anomaly with a **diverging** colormap centred on 0 (`cmap="RdBu_r"`, `vmin=-3`, `vmax=3`): blue colder than usual, red warmer.
Why is a diverging colormap better than `turbo` here?

In [ ]:
### STUDENT: # 🚀 Your code here
### BEGIN SOLUTION
anom = sst_monthly - sst_mean
fig, ax = plt.subplots(figsize=(9, 6), subplot_kw={"projection": proj})
anom.sel(month=3).plot(ax=ax, transform=proj, cmap="RdBu_r", vmin=-3, vmax=3,
                       cbar_kwargs={"label": "SST anomaly (°C)", "shrink": 0.8})
ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.85", zorder=2)
ax.coastlines(resolution="50m", linewidth=0.8, zorder=3)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
gl.top_labels = gl.right_labels = False
ax.set_title("SST anomaly, March 2023 (monthly mean − annual mean)")
plt.show()
# A diverging colormap gives 0 a neutral colour (white) and shows the SIGN of the
# anomaly at a glance; with turbo, 0 would just be some green with no meaning.
### END SOLUTION

**✅ Check**: the whole northern half of the map is colder than the annual mean in March (Caribbean included, ≈ −1.5 °C), with two strong blue patches **down to ≈ −3.5 °C**: offshore of the Gulf of Papagayo and in the Gulf of Panama (≈ 79°W, 7–8°N). The south-west of the map is slightly warmer than average (≈ +0.5 °C). A diverging colormap puts white at 0, so the **sign** of the anomaly is obvious at a glance.

---
## 📋 Recap — your xarray / cartopy cheat-sheet

| I want to… | Code |
|---|---|
| open a file | `ds = xr.open_dataset(path)` |
| see what is inside | `ds`, `ds.sizes`, `list(ds.data_vars)`, `da.attrs` |
| take one variable | `da = ds["analysed_sst"]` |
| select by index / by value | `da.isel(time=0)` / `da.sel(time="2023-03-15")` |
| nearest point | `da.sel(latitude=11, longitude=-87, method="nearest")` |
| a range (check the coordinate order first!) | `da.sel(latitude=slice(8, 12), time="2023-02")` |
| average over a dimension | `da.mean("time")`, `da.mean(["latitude", "longitude"])` |
| monthly climatology | `da.groupby("time.month").mean()` |
| quick look | `da.plot()` |
| map axis | `fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})` |
| field on a map | `da.plot(ax=ax, transform=ccrs.PlateCarree(), vmin=..., vmax=...)` |
| land / coastline / labels | `ax.add_feature(cfeature.LAND)`, `ax.coastlines()`, `ax.gridlines(draw_labels=True)` |
| zoom | `ax.set_extent([W, E, S, N], crs=ccrs.PlateCarree())` |
| several maps, one colorbar | `plt.subplots(n, m, subplot_kw=...)`, `add_colorbar=False`, `fig.colorbar(im, ax=axes)` |
| save | `fig.savefig("file.png", dpi=150, bbox_inches="tight")` (before `plt.show()`) |

**Next: TP1**, where you will use all of this on more files, add vertical sections, and meet the command-line tools NCO and CDO.

## To go further
- xarray tutorial: <https://tutorial.xarray.dev>
- cartopy gallery: <https://scitools.org.uk/cartopy/docs/latest/gallery/index.html>
- Choosing a colormap: <https://matplotlib.org/stable/users/explain/colors/colormaps.html>

<!-- SOLUTION ONLY -->
### 👩‍🏫 Instructor notes

- **Timing** (~1h30–2h for beginners): sections 0–2 ≈ 15 min, section 3 ≈ 25 min, section 4 ≈ 15 min, sections 5–6 ≈ 35 min, debugging ≈ 10 min. Fits the end of TP0 (Monday afternoon) once everyone is logged in, or the first slot on Tuesday before TP1. Natural split if needed: sections 0–4 (xarray) on Monday, 5–7 (maps) on Tuesday morning.
- **Cartopy / Natural Earth**: check on Kabre **before the workshop** that `coastlines(resolution="50m")` works on a compute node. If the nodes have no internet, pre-download the 50m physical/cultural shapefiles into a shared `cartopy.config["data_dir"]`, otherwise 20 people will hit a download error at the same time.
- **Debugging exercise**: the error students see is `ValueError: Axes should be an instance of GeoAxes` — good occasion to show that the last line of the traceback names the problem.
- **Common student mistakes**: positive longitude for 87°W; forgetting `subplot_kw={"projection": ...}`; `savefig` after `plt.show()` (blank PNG); missing `transform=` (invisible with a PlateCarree map, wrong with any other projection — worth mentioning).
- **Overlap with TP1**: TP1 part A.1 (annual mean map) and the latitude trap are introduced here; in TP1 they become a quick reminder.